### 0. Import modules

In [1]:
import stable_baselines3 as sb3
import gym_unbalanced_disk
import gymnasium as gym
import numpy as np
np.random.seed(42)
sb3.common.utils.set_random_seed(42)
SEEDS = [42 + i for i in range(6)]

### 1 Initializing the saving dirs, Prepare the Env, prepare the callback funcitons

In [2]:
"""
Train an A2C/PPO agent on the unbalanced-disk swing-up task using stable-baselines3.

Run from anywhere (the gym_unbalanced_disk import registers the env id):
    python a2c.py

View the training logs with:
    tensorboard --logdir ./tensorboard_logs
"""

import time
from pathlib import Path

import numpy as np
import gymnasium as gym

# Importing gym_unbalanced_disk registers the 'unbalanced-disk-v0' env id with
# gymnasium, so it must be imported before any gym.make(...) call.
import gym_unbalanced_disk  # noqa: F401  (imported for its registration side effect)
import stable_baselines3 as sb3
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import (
    EvalCallback,
    CallbackList,
    BaseCallback,
    StopTrainingOnNoModelImprovement,
    StopTrainingOnRewardThreshold

)

def get_save_dirs(model_name: str):
    # All artifacts (checkpoints, logs) are written next to this script, regardless
    # of the directory the script is launched from.
    try:
        HERE = Path(__file__).resolve().parent
    except NameError:
        # __file__ is undefined in a Jupyter notebook; use the working dir
        HERE = Path.cwd()
    TB_LOG_DIR = HERE / "tensorboard_logs"
    BEST_MODEL_DIR = HERE / f"{model_name}_best"
    EVAL_LOG_DIR = HERE / f"{model_name}_eval_logs"
    CHECKPOINT_DIR = HERE / f"{model_name}_checkpoints"
    FINAL_MODEL_PATH = HERE / f"{model_name}_unbalanced_disk"
    return TB_LOG_DIR, BEST_MODEL_DIR, EVAL_LOG_DIR, CHECKPOINT_DIR, FINAL_MODEL_PATH

TB_LOG_DIR, BEST_MODEL_DIR, EVAL_LOG_DIR, CHECKPOINT_DIR, FINAL_MODEL_PATH = get_save_dirs(f"model")

SIN_COS = True  # whether to use the sin-cos variant of the env (see make_env())
ENV_ID = "unbalanced-disk-v0" if not SIN_COS else "unbalanced-disk-sincos-v0"
# ---------------------------------------------------------------------------
# Environment factory
# ---------------------------------------------------------------------------
# The env id used everywhere below. Switch to the sincos variant by setting
# SIN_COS=True: it exposes a wrap-invariant [sin th, cos th, omega] observation
# (class UnbalancedDisk_sincos), so a policy never falls off the +-pi angle
# discontinuity - important for transfer to the real motor, whose encoder
# reports a wrapped angle.


def make_env(sin_cos= False, robust=False, is_evaluation=False):
    """
    Build a single training/eval environment.

    sin_cos=True  -> 'unbalanced-disk-sincos-v0', obs = [sin th, cos th, omega]
    sin_cos=False -> 'unbalanced-disk-v0',        obs = [th, omega] (raw, unwrapped)
    """
    if sin_cos:
        # Wrap-invariant observation. This is a *separate* registered env
        # (UnbalancedDisk_sincos); the raw v0 env has no obs_as_sin_cos kwarg.
        env = gym.make("unbalanced-disk-sincos-v0", is_evaluation=is_evaluation, dt=0.025, umax=3.0, robust=robust)
    else:
        env = gym.make("unbalanced-disk-v0", is_evaluation=is_evaluation, dt=0.025, umax=3.0, robust=robust)

    env = Monitor(env)

    return env


# ---------------------------------------------------------------------------
# Standalone evaluation (used for a final report after training)
# ---------------------------------------------------------------------------
def evaluate(model, env, num_episodes=10):
    """
    Run the (deterministic) policy for a number of episodes and return the
    mean total reward per episode.
    """
    total_rewards = []
    for _ in range(num_episodes):
        obs, info = env.reset()
        done = False
        episode_reward = 0.0
        while not done:
            action, _states = model.predict(obs, deterministic=True)
            obs, reward, terminated, truncated, info = env.step(action)
            episode_reward += reward
            done = terminated or truncated
        total_rewards.append(episode_reward)
    return float(np.mean(total_rewards))

def evaluate_trained_policy(model_path: Path, num_episodes=10, deterministic=False, robust=False, cos_sin=False):
    if "ppo" in model_path.name:
        model = sb3.PPO.load(str(model_path), device="cpu")
    else:
        model = sb3.A2C.load(str(model_path), device="cpu")
    eval_env = make_env(robust=robust, sin_cos=cos_sin, is_evaluation=True)
    mean_reward = evaluate(model, eval_env, num_episodes=num_episodes)
    print(eval_env.get_wrapper_attr("max_eval_reward") * 0.9)

    print(f"Mean reward over {num_episodes} episodes: {mean_reward:.2f}")
    eval_env.close()

def visualize_trained_policy(model_path: Path, deterministic: bool = False, robust: bool = False, cos_sin: bool = False):
# -----------------------------------------------------------------------
# Visualise the trained policy
# -----------------------------------------------------------------------
# A human-rendered env to watch the swing-up. render_mode="human" opens a
# pygame window.
    if "ppo" in model_path.name:
        model = sb3.PPO.load(str(model_path), device="cpu")
    else:
        model = sb3.A2C.load(str(model_path), device="cpu")
    vis_env = make_env(robust=robust, sin_cos=cos_sin)
    obs, info = vis_env.reset()
    try:
        for _ in range(200):
            action, _states = model.predict(obs, deterministic=deterministic)
            obs, reward, terminated, truncated, info = vis_env.step(action)
            vis_env.render()
            time.sleep(1 / 40)
            if terminated or truncated:
                obs, info = vis_env.reset()   # unpack the (obs, info) tuple
    finally:  # always run, even on Ctrl-C, so the window/resources are released
        vis_env.close()


class RenderEvalCallback(BaseCallback):
    """Every `render_freq` steps, play one deterministic episode in a
    human-rendered env so the current policy can be watched. Kept separate
    from EvalCallback so metric logging (eval_freq) stays untouched."""
    def __init__(self, render_freq=50000, n_episodes=1, max_steps=200, verbose=0, robust=False, sin_cos=False):
        super().__init__(verbose)
        self.render_freq = render_freq
        self.n_episodes = n_episodes
        self.max_steps = max_steps
        self.robust = robust
        self.sin_cos = sin_cos

    def _on_step(self) -> bool:
        # n_calls increments once per _on_step; with a single env this == timesteps.
        # For multiple envs use self.num_timesteps instead.
        if self.n_calls % self.render_freq == 0:
            vis_env = make_env(robust=self.robust, sin_cos=self.sin_cos)
            try:
                for _ in range(self.n_episodes):
                    obs, info = vis_env.reset()
                    for _ in range(self.max_steps):
                        action, _ = self.model.predict(obs, deterministic=True)
                        obs, reward, terminated, truncated, info = vis_env.step(action)
                        vis_env.render()
                        time.sleep(1 / 40)
                        if terminated or truncated:
                            break
            finally:
                vis_env.close()
        return True


def get_callbacks(eval_env, robust=False, sin_cos=False):

    # Early stopping: stop training once the eval reward has not produced a new best for 3 consecutive evals (i.e. 75k steps) - this is a sign of convergence.
    stop_callback = StopTrainingOnNoModelImprovement(
        max_no_improvement_evals=3,   # ~3 evals * 25k = 75k steps of no new best
        min_evals=10,                 # don't even consider stopping before 250k steps
        verbose=1,
    )
    
    # Stop when we reach 85% of the max reward (for evaluation env). This is a more aggressive stopping criterion that doesn't wait for multiple evaluations with no improvement, but it relies on having a good estimate of the max reward.
    stop_reward_callback = StopTrainingOnRewardThreshold(
        reward_threshold=eval_env.get_wrapper_attr("max_eval_reward") * 0.9,  # 95% of max reward
        verbose=1,
    )

    # Evaluate agent every 25k steps.
    eval_callback = EvalCallback(
        eval_env,
        best_model_save_path=str(BEST_MODEL_DIR),
        log_path=str(EVAL_LOG_DIR),
        eval_freq=25000,       # run an evaluation every 25k training steps
        n_eval_episodes=5,
        deterministic=True,
        render=False,
        callback_after_eval=stop_callback,
        callback_on_new_best=stop_reward_callback,
    )

    # Metric eval + early stopping, plus a watchable rendered rollout every 200k steps.
    callbacks = CallbackList([eval_callback])

    return callbacks

def get_model(model_name: str, env, callbacks = None):
    if model_name == "a2c":
        model = sb3.A2C(
            "MlpPolicy",
            env,
            
            verbose=0,
            tensorboard_log=str(TB_LOG_DIR),
        )
    elif model_name == "ppo":
        model = sb3.PPO(
            "MlpPolicy",
            env,
            verbose=0,
            device="cpu",
            tensorboard_log=str(TB_LOG_DIR),
        )

    return model

In [3]:
# ---------------------------------------------------------------------------
# Per-model run metrics (training time, steps trained, final mean reward, seed)
# ---------------------------------------------------------------------------
# Every training run appends one row to this CSV so all models/seeds accumulate
# in a single comparable table. Read it back with pandas.read_csv(RESULTS_CSV);
# analyze_results.py aggregates the rows over seeds into mean +/- std.
import csv
from datetime import datetime

RESULTS_CSV = Path.cwd() / "training_results.csv"

# Fixed column order so the header is stable across runs / algorithms / seeds.
RESULTS_FIELDS = [
    "timestamp", "name", "algo", "seed", "base_reward", "robust", "sin_cos",
    "num_timesteps", "train_time_s", "train_time_min", "mean_reward",
]

def log_run_metrics(name, algo, num_timesteps, train_time_s, mean_reward,
                    seed=None, base_reward=None, robust=None, sin_cos=None):
    """Append a single run's metrics to RESULTS_CSV (writing the header once)."""
    row = {
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "name": name,
        "algo": algo,
        "seed": seed,
        "base_reward": base_reward,
        "robust": robust,
        "sin_cos": sin_cos,
        "num_timesteps": int(num_timesteps),
        "train_time_s": round(train_time_s, 1),
        "train_time_min": round(train_time_s / 60, 2),
        "mean_reward": round(float(mean_reward), 3),
    }
    write_header = not RESULTS_CSV.exists()
    with RESULTS_CSV.open("a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=RESULTS_FIELDS)
        if write_header:
            w.writeheader()
        w.writerow(row)
    print(f"Logged -> {RESULTS_CSV.name}: {name} seed={seed} "
          f"steps={row['num_timesteps']} time={row['train_time_min']}min "
          f"reward={row['mean_reward']}")
    return row


In [ ]:
# ---------------------------------------------------------------------------
# Plot a trained policy's behaviour: states (theta, omega) and action over time
# ---------------------------------------------------------------------------
# Unlike visualize_trained_policy() (which opens a live pygame window), this
# runs one head-less rollout, records the *true* physical state each step, and
# draws static plots suitable for the report.
import matplotlib.pyplot as plt


def plot_trained_policy(model_path, deterministic=True, robust=False, cos_sin=False,
                        n_steps=200, save_path=None, show=True):
    """Roll out one episode and plot theta, omega, the chosen action, and the
    per-step reward against time. Returns the recorded arrays as a dict."""
    model_path = Path(model_path)
    if "ppo" in model_path.name:
        model = sb3.PPO.load(str(model_path), device="cpu")
    else:
        model = sb3.A2C.load(str(model_path), device="cpu")

    # Use the evaluation env so the reward trace matches the reported metric.
    env = make_env(robust=robust, sin_cos=cos_sin, is_evaluation=True)
    dt = env.get_wrapper_attr("dt")
    umax = env.get_wrapper_attr("umax")

    obs, info = env.reset()
    ts, thetas, omegas, actions, applied_u, rewards = [], [], [], [], [], []
    for k in range(n_steps):
        action, _ = model.predict(obs, deterministic=deterministic)
        obs, reward, terminated, truncated, info = env.step(action)
        ts.append(k * dt)
        # True physical state (the policy only sees a noisy/encoded version of it).
        thetas.append(float(env.get_wrapper_attr("th")))
        omegas.append(float(env.get_wrapper_attr("omega")))
        actions.append(float(np.asarray(action).item()))        # what the policy commanded
        applied_u.append(float(np.asarray(env.get_wrapper_attr("u")).item()))  # after robust noise + clip
        rewards.append(float(reward))
        if terminated or truncated:
            break
    env.close()

    ts, thetas, omegas = map(np.asarray, (ts, thetas, omegas))
    actions, applied_u, rewards = map(np.asarray, (actions, applied_u, rewards))

    fig, axes = plt.subplots(4, 1, sharex=True, figsize=(9, 9))

    # theta: raw angle climbs toward +/- pi (upright) during the swing-up.
    axes[0].plot(ts, thetas, c="C0")
    axes[0].axhline(np.pi, ls="--", c="g", lw=1, label=r"upright ($\pm\pi$)")
    axes[0].axhline(-np.pi, ls="--", c="g", lw=1)
    axes[0].set_ylabel(r"$\theta$ [rad]")
    axes[0].legend(loc="upper right")

    axes[1].plot(ts, omegas, c="C1")
    axes[1].set_ylabel(r"$\omega$ [rad/s]")

    # Chosen action (step plot); show the applied torque too if robust noise differs.
    axes[2].plot(ts, actions, c="C2", drawstyle="steps-post", label="chosen")
    if robust:
        axes[2].plot(ts, applied_u, c="C4", drawstyle="steps-post", alpha=0.5, label="applied")
        axes[2].legend(loc="upper right")
    axes[2].axhline(umax, ls=":", c="k", lw=1)
    axes[2].axhline(-umax, ls=":", c="k", lw=1)
    axes[2].set_ylabel("action $u$")

    axes[3].plot(ts, rewards, c="C3")
    axes[3].set_ylabel("reward")
    axes[3].set_xlabel("time [s]")

    fig.suptitle(f"{model_path.stem}   (sum reward = {rewards.sum():.1f})")
    fig.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"Saved plot to {save_path}")
    if show:
        plt.show()
    else:
        plt.close(fig)

    return dict(t=ts, theta=thetas, omega=omegas,
                action=actions, applied_u=applied_u, reward=rewards)
"""    dict(base_reward=True,  sin_cos=False, robust=False, suffix="baseR"),
    dict(base_reward=False, sin_cos=False, robust=False, suffix="tunedR"),"""

## 2. Running experiments

 the models for the Advanced RL chapter of the report. The current reward structure in UnbalancedDisc.py represents the Tuned_R from the rapport, therefor running the following cells wont reproduce experiment 1 unless the reward function is explicitely changed.


### 2.1  Experiment 1/2:

#### 2.1.1 A2C training
Below is a script that allows for base training of the experiment 1 and 2 A2C models

In [ ]:

# Separate environments for training and for periodic evaluation, so eval
# episodes do not interfere with the training rollouts.

A2C_CONFIGS = [

    dict(base_reward=True,  sin_cos=True,  robust=False, suffix="baseR_sincos"),
    dict(base_reward=False, sin_cos=True,  robust=False, suffix="tunedR_sincos"),
    dict(base_reward=True,  sin_cos=True,  robust=True,  suffix="baseR_sincos_robust"),
    dict(base_reward=False, sin_cos=True,  robust=True,  suffix="tunedR_sincos_robust"),
]


def train_a2c(robust=False, sin_cos=False, suffix="", base_reward=False, seed=0, visualize=False):
    # A2C with the default MLP policy. device="cpu" is correct here: the network
    # is tiny and the bottleneck is the scipy ODE solve inside each env step.
    # Seed every global RNG (python random, numpy, torch) from the single seed.
    sb3.common.utils.set_random_seed(seed)

    TB_LOG_DIR, BEST_MODEL_DIR, EVAL_LOG_DIR, CHECKPOINT_DIR, FINAL_MODEL_PATH = get_save_dirs(f"a2c_{suffix}_s{seed}")
    train_env = make_env(sin_cos=sin_cos, robust=robust, is_evaluation=base_reward)
    # Same nominal base eval env (robust=False) as PPO, so returns are comparable.
    eval_env = make_env(sin_cos=sin_cos, robust=False, is_evaluation=True)
    # Seed the per-env RNG (initial state) and the action spaces.
    train_env.reset(seed=seed); train_env.action_space.seed(seed)
    eval_env.reset(seed=seed); eval_env.action_space.seed(seed)

    model = get_model("a2c", train_env)
    callbacks = get_callbacks(eval_env, robust=robust, sin_cos=sin_cos)

    t0 = time.time()
    model.learn(
        total_timesteps=1000000,
        tb_log_name=f"a2c_{suffix}_s{seed}",
        progress_bar=True,
        callback=callbacks,
    )
    train_time_s = time.time() - t0
    # Capture actual steps trained before reloading the best model resets it.
    num_timesteps = model.num_timesteps

    model.save(str(FINAL_MODEL_PATH))

    # Prefer the best checkpoint for the final report; fall back to the final model.
    best_model_path = BEST_MODEL_DIR / "best_model.zip"
    if best_model_path.exists():
        model = sb3.A2C.load(str(best_model_path), device="cpu")

    eval_env.reset(seed=seed)  # reset RNG so the final report is reproducible
    mean_reward = evaluate(model, eval_env, num_episodes=10)

    log_run_metrics(
        name=f"a2c_{suffix}", algo="a2c",
        num_timesteps=num_timesteps, train_time_s=train_time_s,
        mean_reward=mean_reward, seed=seed,
        base_reward=base_reward, robust=robust, sin_cos=sin_cos,
    )

    train_env.close()
    eval_env.close()

    if visualize:  # off by default so overnight/headless sweeps don't block on pygame
        visualize_trained_policy(Path(f"{FINAL_MODEL_PATH}.zip"), robust=robust, cos_sin=sin_cos)


for cfg in A2C_CONFIGS:
    for seed in SEEDS:
        train_a2c(seed=seed, **cfg)


/home/nemo/Documents/GitHub/IML/DesignProject/.venv/lib/python3.12/site-packages/gymnasium/utils/passive_env_checker.py:134: UserWarning: WARN: The obs returned by the `reset()` method was expecting numpy array dtype to be float32, actual type: float64
  logger.warn(
/home/nemo/Documents/GitHub/IML/DesignProject/.venv/lib/python3.12/site-packages/gymnasium/utils/passive_env_checker.py:158: UserWarning: WARN: The obs returned by the `reset()` method is not within the observation space.
  logger.warn(f"{pre} is not within the observation space.")


Output()

/home/nemo/Documents/GitHub/IML/DesignProject/.venv/lib/python3.12/site-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run A2C on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


/home/nemo/Documents/GitHub/IML/DesignProject/.venv/lib/python3.12/site-packages/gymnasium/utils/passive_env_checke
r.py:134: UserWarning: WARN: The obs returned by the `step()` method was expecting numpy array dtype to be float32,
actual type: float64
  logger.warn(

/home/nemo/Documents/GitHub/IML/DesignProject/.venv/lib/python3.12/site-packages/gymnasium/utils/passive_env_checke
r.py:158: UserWarning: WARN: The obs returned by the `step()` method is not within the observation space.
  logger.warn(f"{pre} is not within the observation space.")

#### 2.1.2 PPO training cell

In [ ]:
# (base_reward, sin_cos, robust) with names that now MATCH the booleans
# (the earlier sincos names were swapped relative to the flag actually passed).
PPO_CONFIGS = [
    dict(base_reward=True,  sin_cos=False, robust=False, name_suffix="baseR"),
    dict(base_reward=False, sin_cos=False, robust=False, name_suffix="tunedR"),
    dict(base_reward=True,  sin_cos=True,  robust=False, name_suffix="baseR_sincos"),
    dict(base_reward=False, sin_cos=True,  robust=False, name_suffix="tunedR_sincos"),
    dict(base_reward=True,  sin_cos=True,  robust=True,  name_suffix="baseR_sincos_robust"),
    dict(base_reward=False, sin_cos=True,  robust=True,  name_suffix="tunedR_sincos_robust"),
]


def train_ppo(base_reward=False, robust=False, sin_cos=False, name_suffix="", seed=0, visualize=False):
    # Seed every global RNG (python random, numpy, torch) from the single seed.
    # This makes the policy weight init, action sampling, and the env's global-
    # numpy noise (actuator/observation noise) reproducible for this run.
    sb3.common.utils.set_random_seed(seed)

    # Per-seed save dirs so seeds don't overwrite each other's checkpoints.
    TB_LOG_DIR, BEST_MODEL_DIR, EVAL_LOG_DIR, CHECKPOINT_DIR, FINAL_MODEL_PATH = get_save_dirs(f"ppo_{name_suffix}_s{seed}")
    train_env = make_env(sin_cos=sin_cos, robust=robust, is_evaluation=base_reward)
    # Both algos are evaluated on the SAME nominal base env (robust=False) so the
    # reported return is comparable across configs.
    eval_env = make_env(sin_cos=sin_cos, robust=False, is_evaluation=True)
    # Seed the per-env RNG (initial state via env.np_random) and the action spaces.
    train_env.reset(seed=seed); train_env.action_space.seed(seed)
    eval_env.reset(seed=seed); eval_env.action_space.seed(seed)

    # Training
    model = get_model("ppo", train_env)
    callbacks = get_callbacks(eval_env, robust=robust, sin_cos=sin_cos)
    t0 = time.time()
    model.learn(
            total_timesteps=1000000,
            tb_log_name=f"ppo_{name_suffix}_s{seed}",
            callback=callbacks,
            progress_bar=True,
        )
    train_time_s = time.time() - t0
    # Capture the actual steps trained now: with early stopping this is < 1M,
    # and reloading the best model below would reset model.num_timesteps.
    num_timesteps = model.num_timesteps

    model.save(str(FINAL_MODEL_PATH))

    # EVALUATION on the best checkpoint (fall back to final if none saved).
    best_model_path = BEST_MODEL_DIR / "best_model.zip"
    if best_model_path.exists():
        model = sb3.PPO.load(str(best_model_path), device="cpu")

    eval_env.reset(seed=seed)  # reset RNG so the final report is reproducible
    mean_reward = evaluate(model, eval_env, num_episodes=10)

    # Persist this run's metrics (incl. seed) to the CSV.
    log_run_metrics(
        name=f"ppo_{name_suffix}", algo="ppo",
        num_timesteps=num_timesteps, train_time_s=train_time_s,
        mean_reward=mean_reward, seed=seed,
        base_reward=base_reward, robust=robust, sin_cos=sin_cos,
    )

    train_env.close()
    eval_env.close()

    if visualize:  # off by default so overnight/headless sweeps don't block on pygame
        visualize_trained_policy(Path(f"{FINAL_MODEL_PATH}.zip"), deterministic=False, robust=robust, cos_sin=sin_cos)


for cfg in PPO_CONFIGS:
    for seed in SEEDS:
        train_ppo(seed=seed, **cfg)